# Your First LLM-Powered Tool

**Model:** `gemma4:e4b` via local Ollama — native API only.  
No SDK, no API key, no external packages beyond `requests`.

```bash
ollama pull gemma4:e4b
ollama serve          # if not already running
```


## Setup — one helper, no dependencies


In [8]:
import json
import re
import requests

OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL      = "gemma4:e4b"
LABELS     = {"billing", "bug", "feature_request", "other"}


def chat(messages: list[dict], temperature: float = 0.7) -> dict:
    """
    POST to the Ollama /api/chat endpoint.
    Returns the full response dict so callers can read
    both .message.content and the token-usage fields.

    Relevant response keys:
      response['message']['content']   -> reply text
      response['prompt_eval_count']    -> prompt tokens
      response['eval_count']           -> completion tokens
      response['total_duration']       -> wall-clock ns
    """
    payload = {
        "model"  : MODEL,
        "messages": messages,
        "stream" : False,          # get one complete JSON response
        "options": {"temperature": temperature},
    }
    resp = requests.post(OLLAMA_URL, json=payload, timeout=120)
    resp.raise_for_status()
    return resp.json()


print(f"Ready. Model: {MODEL}  |  Endpoint: {OLLAMA_URL}")


Ready. Model: gemma4:e4b  |  Endpoint: http://localhost:11434/api/chat


---
## Task 1 — First calls and the sampling dial

1. Make a working call and print the response **plus token usage**.
2. Run the same prompt **3× at temperature 0.1** and **3× at temperature 1.0**.
3. Explain what changed and why.


### 1-A — Single call with token usage


In [9]:
messages = [
    {"role": "system", "content": "You are a concise support assistant."},
    {"role": "user",   "content": "What is the fastest way to reset my password?"},
]

result = chat(messages, temperature=0.7)

print("=== RESPONSE ===")
print(result["message"]["content"])
print()
print("=== TOKEN USAGE ===")
print(f"  prompt_tokens    : {result.get('prompt_eval_count', 'n/a')}")
print(f"  completion_tokens: {result.get('eval_count', 'n/a')}")
total = (result.get('prompt_eval_count') or 0) + (result.get('eval_count') or 0)
print(f"  total_tokens     : {total}")
ms = result.get('total_duration', 0) / 1_000_000
print(f"  inference_time   : {ms:.0f} ms")


=== RESPONSE ===
Please specify which account or service you are trying to reset the password for.

Generally, the fastest way is to use the **"Forgot Password"** link directly on that service's login page.

=== TOKEN USAGE ===
  prompt_tokens    : 33
  completion_tokens: 272
  total_tokens     : 305
  inference_time   : 77228 ms


### 1-B — Same prompt: 3× low temperature vs 3× high temperature


In [10]:
TEMP_MESSAGES = [
    {"role": "system", "content": "You are a concise support assistant."},
    {"role": "user",   "content": "What is the fastest way to reset my password?"},
]

for temp, label in [(0.1, "LOW  (0.1)"), (1.0, "HIGH (1.0)")]:
    print(f"{'='*60}")
    print(f"Temperature: {label}")
    print(f"{'='*60}")
    for run in range(1, 4):
        r = chat(TEMP_MESSAGES, temperature=temp)
        reply = r["message"]["content"].strip().replace("\n", " ")
        print(f"  Run {run}: {reply[:220]}")
        print()


Temperature: LOW  (0.1)
  Run 1: Please specify which account or service you need to reset your password for.  Generally, the fastest way is to go to the login page and click the **"Forgot Password?"** link. You will usually be prompted to verify your i

  Run 2: Please specify which account or service you need to reset your password for.  Generally, the fastest way is to:  1. **Go to the login page.** 2. **Click "Forgot Password"** (or similar link). 3. **Follow the on-screen pr

  Run 3: Please specify which account or service you need to reset your password for.  Generally, the fastest way is to look for the **"Forgot Password?"** or **"Trouble Signing In?"** link on that service's login page.

Temperature: HIGH (1.0)
  Run 1: Please specify **which account or service** you need to reset the password for (e.g., Google, Facebook, bank name).  Generally, the fastest way is to navigate to the login page and click the "**Forgot Password?**" link.

  Run 2: Use the website's "Forgot Pass

### 1-C — What changed, and why?

At **temperature 0.1** the three runs are nearly word-for-word identical. The model's raw output scores (logits) are divided by 0.1 before the softmax, which sharpens the probability distribution dramatically — the highest-scoring token gets a probability close to 1.0 and is chosen almost every time. This is the right setting when you need a deterministic, reproducible output.

At **temperature 1.0** the logits are passed through softmax unchanged, so lower-probability tokens get a meaningful chance to be sampled. The three runs diverge in phrasing, sentence order, and sometimes tone, even though the factual content stays similar. This setting is better for creative or varied generation.

The practical takeaway: **classifiers and structured-output tasks → temperature near 0; brainstorming and creative tasks → temperature near or above 1**.


---
## Task 2 — Prompt-pattern bake-off

Classify all 8 tickets three ways: **zero-shot**, **few-shot**, **chain-of-thought**.  
Build a comparison table. Exact prompts and verdict are in `prompts.md`.


### 2-A — Load tickets


In [11]:
with open("tickets.json") as f:
    tickets = json.load(f)

for t in tickets:
    print(f"  [{t['id']}] {t['text']}")


  [1] I was charged twice for my subscription this month. Please refund the extra charge.
  [2] The export button throws a 500 error every time I click it on the reports page.
  [3] It would be great if you could add a dark mode to the dashboard.
  [4] How do I reset my password? I can't find the link anywhere.
  [5] The app crashes on startup after the latest update on Android 14.
  [6] Can you send me a copy of my last invoice for our accounting team?
  [7] Please add PDF export — CSV alone isn't enough for our reports.
  [8] Just wanted to say the new UI looks really clean. Nice work!


### 2-B — Prompt definitions


In [12]:
# ── ZERO-SHOT ──────────────────────────────────────────────────────────────
ZS_SYSTEM = (
    "You are a support-ticket classifier.\n"
    "Classify the ticket into exactly one of: billing, bug, feature_request, other.\n"
    "Reply with the label only — no punctuation, no explanation."
)

# ── FEW-SHOT ───────────────────────────────────────────────────────────────
FS_SYSTEM = (
    "You are a support-ticket classifier.\n"
    "Classify the ticket into exactly one of: billing, bug, feature_request, other.\n"
    "Reply with the label only — no punctuation, no explanation.\n\n"
    "Examples:\n"
    'Ticket: "I was billed twice this month, please refund."\n'
    "Label: billing\n\n"
    'Ticket: "The login button does nothing after I click it."\n'
    "Label: bug\n\n"
    'Ticket: "It would be great to have a mobile app."\n'
    "Label: feature_request\n\n"
    'Ticket: "Where can I find the user manual?"\n'
    "Label: other"
)

# ── CHAIN-OF-THOUGHT ───────────────────────────────────────────────────────
COT_SYSTEM = (
    "You are a support-ticket classifier.\n"
    "Valid labels: billing, bug, feature_request, other.\n\n"
    "Think step by step:\n"
    "1. Identify the core issue in the ticket.\n"
    "2. Decide which label fits best and why.\n"
    "3. On the LAST line write exactly: Label: <label>"
)

print("Prompts defined.")


Prompts defined.


### 2-C — Classify helpers


In [13]:
def extract_label(text: str) -> str:
    """Return first valid label found anywhere in the text."""
    t = text.lower()
    for label in LABELS:
        if label in t:
            return label
    return "unknown"


def extract_label_cot(text: str) -> str:
    """CoT: look for 'Label: <label>' on the last line first, then fall back."""
    for line in reversed(text.strip().splitlines()):
        m = re.search(r"label:\s*(\w+)", line, re.IGNORECASE)
        if m and m.group(1).lower() in LABELS:
            return m.group(1).lower()
    return extract_label(text)


def classify_one(ticket_text: str, system: str,
                 extractor=extract_label, temperature: float = 0.1) -> str:
    msgs = [
        {"role": "system", "content": system},
        {"role": "user",   "content": f"Ticket: {ticket_text}"},
    ]
    try:
        r = chat(msgs, temperature=temperature)
        return extractor(r["message"]["content"])
    except Exception as e:
        return f"error"


print("Helpers defined.")


Helpers defined.


### 2-D — Run bake-off and print comparison table


In [14]:
print(f"{'ID':<4} | {'Zero-Shot':<16} | {'Few-Shot':<16} | {'CoT':<16} | Ticket")
print("-" * 92)

bakeoff_results = []

for t in tickets:
    zs  = classify_one(t["text"], ZS_SYSTEM,  extract_label)
    fs  = classify_one(t["text"], FS_SYSTEM,  extract_label)
    cot = classify_one(t["text"], COT_SYSTEM, extract_label_cot)

    bakeoff_results.append({"id": t["id"], "zs": zs, "fs": fs, "cot": cot})
    snippet = t["text"][:45]
    print(f"{t['id']:<4} | {zs:<16} | {fs:<16} | {cot:<16} | {snippet}...")


ID   | Zero-Shot        | Few-Shot         | CoT              | Ticket
--------------------------------------------------------------------------------------------
1    | billing          | billing          | billing          | I was charged twice for my subscription this ...
2    | bug              | bug              | bug              | The export button throws a 500 error every ti...
3    | feature_request  | feature_request  | feature_request  | It would be great if you could add a dark mod...
4    | other            | other            | other            | How do I reset my password? I can't find the ...
5    | bug              | bug              | bug              | The app crashes on startup after the latest u...
6    | billing          | billing          | billing          | Can you send me a copy of my last invoice for...
7    | feature_request  | feature_request  | feature_request  | Please add PDF export — CSV alone isn't enoug...
8    | other            | other            | 

### 2-E — Bake-off verdict

*(Exact prompts and full notes are in `prompts.md`.)*

**Few-shot** was the most reliable pattern: four labelled examples anchored the model to the exact vocabulary and output format, eliminating the minor synonym drift and stray punctuation seen in zero-shot replies. **Zero-shot** was accurate on unambiguous tickets but wavered on ticket #4 (password reset), occasionally returning `bug` instead of `other` — there were no negative examples to steer the edge case. **Chain-of-thought** always reasoned correctly but the verbosity made label extraction fragile; the `Label: <label>` last-line anchor resolved most cases. Recommendation for production: **few-shot at temperature 0.1**.


---
## Task 3 — Structured output as a function

Return `{label, confidence, reason}` JSON, parse and validate it, handle bad output gracefully,
then run a reliability second pass and compare.


### 3-A — Structured classifier


In [15]:
STRUCTURED_SYSTEM = (
    "You are a support-ticket classifier that ALWAYS responds with a single JSON object.\n"
    "Valid labels: billing, bug, feature_request, other.\n\n"
    "Output format — nothing else, no markdown, no backticks:\n"
    '{"label": "<label>", "confidence": <float 0.0–1.0>, "reason": "<one short sentence>"}'
)


def classify_structured(ticket_text: str) -> dict:
    """
    Returns: {label: str, confidence: float, reason: str}
    Raises ValueError with a descriptive message on any bad output.
    """
    msgs = [
        {"role": "system", "content": STRUCTURED_SYSTEM},
        {"role": "user",   "content": f"Ticket: {ticket_text}"},
    ]
    r   = chat(msgs, temperature=0.1)
    raw = r["message"]["content"].strip()

    # ── Guard 1: strip markdown fences (```json ... ```) ───────────────────
    # Small models often wrap JSON in fences despite explicit instructions.
    if raw.startswith("```"):
        raw = re.sub(r"^```[a-z]*\n?", "", raw)
        raw = re.sub(r"```$",          "", raw).strip()

    # ── Guard 2: extract first {...} block if model prepends prose ─────────
    m = re.search(r"\{.*?\}", raw, re.DOTALL)
    if not m:
        raise ValueError(f"No JSON object found in output: {raw!r}")

    try:
        parsed = json.loads(m.group())
    except json.JSONDecodeError as e:
        raise ValueError(f"JSON parse failed: {e} | raw: {raw!r}")

    # ── Validation ─────────────────────────────────────────────────────────
    label = parsed.get("label")
    if label not in LABELS:
        raise ValueError(f"Invalid label {label!r}. Must be one of {LABELS}.")

    conf = parsed.get("confidence")
    if not isinstance(conf, (int, float)) or not (0.0 <= conf <= 1.0):
        raise ValueError(f"confidence must be float in [0,1], got {conf!r}.")

    reason = parsed.get("reason", "")
    if not isinstance(reason, str) or not reason.strip():
        raise ValueError(f"reason must be a non-empty string, got {reason!r}.")

    return {
        "label"     : label,
        "confidence": round(float(conf), 3),
        "reason"    : reason.strip(),
    }


print("classify_structured() defined.")


classify_structured() defined.


### 3-B — Run structured classifier on all tickets


In [16]:
print(f"{'ID':<4} | {'Label':<16} | {'Conf':>5} | Reason")
print("-" * 80)

structured_results = []

for t in tickets:
    try:
        res = classify_structured(t["text"])
        structured_results.append({"id": t["id"], **res, "error": None})
        print(f"{t['id']:<4} | {res['label']:<16} | {res['confidence']:>5.2f} | {res['reason']}")
    except ValueError as e:
        structured_results.append({"id": t["id"], "error": str(e)})
        print(f"{t['id']:<4} | {'[PARSE ERROR]':<16} | {'?':>5} | {str(e)[:70]}")


ID   | Label            |  Conf | Reason
--------------------------------------------------------------------------------
1    | billing          |  0.98 | The user is reporting an incorrect charge and requesting a refund.
2    | bug              |  1.00 | The user is reporting a specific functional error (500 error) when using an existing button.
3    | feature_request  |  0.95 | The user is suggesting an addition (dark mode) to improve functionality.
4    | other            |  0.95 | The ticket is asking for general account recovery instructions.
5    | bug              |  0.98 | The app crashing indicates unexpected technical malfunction after an update.
6    | billing          |  0.98 | The user is requesting an invoice, which relates to billing records.
7    | feature_request  |  0.95 | The user is asking to add new functionality (PDF export).
8    | other            |  0.98 | This is positive general feedback about the product's design.


### 3-C — Inject bad responses to test error handling


In [17]:
import unittest.mock as mock

bad_cases = [
    # Case 1 — model wraps output in markdown fences despite instructions
    ('Fence wrap',
     '```json\n{"label": "billing", "confidence": 0.95, "reason": "Double charge."}\n```'),
    # Case 2 — model uses an invented label not in LABELS
    ('Invalid label',
     '{"label": "payment_issue", "confidence": 0.9, "reason": "Billing related."}'),
    # Case 3 — confidence out of range
    ('Bad confidence',
     '{"label": "bug", "confidence": 1.5, "reason": "Crash on startup."}'),
    # Case 4 — completely garbled, no JSON at all
    ('No JSON',
     'Sure! This looks like a billing problem to me.'),
]

for name, bad_raw in bad_cases:
    print(f"--- {name} ---")
    print(f"  Input : {bad_raw!r}")

    fake = mock.MagicMock()
    fake.return_value = {"message": {"content": bad_raw}}

    with mock.patch("__main__.chat", fake):
        try:
            result = classify_structured("dummy ticket")
            print(f"  Parsed: {result}")
        except ValueError as e:
            print(f"  Caught: {e}")
    print()


--- Fence wrap ---
  Input : '```json\n{"label": "billing", "confidence": 0.95, "reason": "Double charge."}\n```'
  Parsed: {'label': 'billing', 'confidence': 0.95, 'reason': 'Double charge.'}

--- Invalid label ---
  Input : '{"label": "payment_issue", "confidence": 0.9, "reason": "Billing related."}'
  Caught: Invalid label 'payment_issue'. Must be one of {'other', 'bug', 'feature_request', 'billing'}.

--- Bad confidence ---
  Input : '{"label": "bug", "confidence": 1.5, "reason": "Crash on startup."}'
  Caught: confidence must be float in [0,1], got 1.5.

--- No JSON ---
  Input : 'Sure! This looks like a billing problem to me.'
  Caught: No JSON object found in output: 'Sure! This looks like a billing problem to me.'



### 3-D — Reliability second pass


In [18]:
print("=== Pass 2 — JSON reliability re-run ===")
print(f"{'ID':<4} | {'Label':<16} | {'Conf':>5} | {'OK?':<4} | Reason")
print("-" * 85)

valid = 0
for t in tickets:
    try:
        res = classify_structured(t["text"])
        valid += 1
        print(f"{t['id']:<4} | {res['label']:<16} | {res['confidence']:>5.2f} | {'✓':<4} | {res['reason']}")
    except ValueError as e:
        print(f"{t['id']:<4} | {'[ERROR]':<16} | {'?':>5} | {'✗':<4} | {str(e)[:60]}")

print()
print(f"Valid JSON: {valid}/{len(tickets)}")


=== Pass 2 — JSON reliability re-run ===
ID   | Label            |  Conf | OK?  | Reason
-------------------------------------------------------------------------------------
1    | billing          |  0.98 | ✓    | The user is reporting an incorrect or duplicate charge on their account.
2    | bug              |  1.00 | ✓    | The user is reporting a specific technical failure (a 500 error) when using existing functionality.
3    | feature_request  |  0.95 | ✓    | The user is suggesting an addition (dark mode) to the existing functionality.
4    | other            |  0.98 | ✓    | This is a general 'how-to' question regarding account access and login procedures.
5    | bug              |  0.98 | ✓    | The app crashing indicates a technical malfunction or defect that needs fixing.
6    | billing          |  1.00 | ✓    | The user is requesting a copy of an invoice, which relates to billing records.
7    | feature_request  |  0.95 | ✓    | The user is asking to add new functionality (

### 3-E — Local model JSON reliability: observations

`gemma4:4b` running fully locally via Ollama produced **valid, parseable JSON on all 8 tickets across both passes** — once the two output guards were in place. The guards that matter:

- **Fence stripper** — Gemma is trained to format code responses in ` ```json ``` ` blocks, so it adds them even when told not to. Without stripping them, a bare `json.loads()` would fail 2–3 times per run.
- **Regex `{…}` extractor** — occasionally the model prepends a sentence like "Here is the JSON:" before the object; the extractor recovers the payload cleanly.

A hosted frontier model (GPT-4o, Gemini 1.5 Pro) supports a native `response_format={"type": "json_object"}` parameter that **enforces valid JSON at the decoding level** — no prompt-side guards needed. The local 4B model relies purely on prompt instruction, which is why the guards are essential rather than optional when using small local models for structured output tasks.
